# Notebook 04 — Train Bitemporal U-Net (Early Fusion)

Trains a U-Net with early fusion for bitemporal forest disturbance detection:
- **Input**: 12 channels (T1: B02/B03/B04/B8A/B11/B12 + T2: same 6 bands concatenated)
- **Output**: binary change mask (1 = forest loss, 0 = no change)
- **Encoder**: ResNet-34 pretrained on ImageNet
- **Loss**: BCE + Dice

**Environment**: Google Colab T4 GPU  
**Runtime**: ~2-3 hours

In [ ]:
# === Cell 1: Install dependencies (Colab only) ===
import subprocess, sys
try:
    import segmentation_models_pytorch as smp
    print(f'smp {smp.__version__} already installed')
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'segmentation-models-pytorch', 'albumentations'], check=True)
    print('Installed smp + albumentations')

In [ ]:
# === Cell 2: Imports & config ===
import json, random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from pathlib import Path
from tqdm import tqdm

from google.colab import drive
drive.mount('/content/drive')

BASE        = Path('/content/drive/MyDrive/GeoAI/forest-disturbance')
PATCHES_T1  = BASE / 'data/patches/images_t1'
PATCHES_T2  = BASE / 'data/patches/images_t2'
PATCHES_MSK = BASE / 'data/patches/masks'
MODELS_DIR  = BASE / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42
BATCH_SIZE = 8
N_EPOCHS   = 50
LR         = 1e-4
PATCH_SIZE = 224   # center-crop from 256

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Device: {DEVICE}')
print(f'Patches T1: {len(list(PATCHES_T1.glob("*.npy")))}')

In [ ]:
# === Cell 2b: Verify patches in Drive before copying ===
drive_patches = BASE / 'data/patches'

counts = {}
for sub in ['images_t1', 'images_t2', 'masks']:
    counts[sub] = len(list((drive_patches / sub).glob('*.npy')))

split_ok = (drive_patches / 'split.json').exists()

print('Drive patch counts:')
for sub, n in counts.items():
    status = 'OK' if n == 3224 else f'MISSING {3224 - n} files'
    print(f'  {sub}: {n}  [{status}]')
print(f'  split.json: {split_ok}')

all_ok = all(n == 3224 for n in counts.values()) and split_ok
if all_ok:
    print('\nAll patches present in Drive — ready to train.')
else:
    raise RuntimeError('Patches incomplete in Drive. Wait for sync to finish before continuing.')

In [ ]:
# === Cell 3: Copy patches to /content/ for fast I/O ===
import subprocess, shutil, json

LOCAL = Path('/content/patches')

# Load split from Drive to know expected count (LOCAL may not exist yet)
with open(BASE / 'data/patches/split.json') as f:
    SPLIT = json.load(f)

def _count(folder):
    return len(list(folder.glob('*.npy'))) if folder.exists() else 0

expected = len(SPLIT['train']) + len(SPLIT['val'])
actual   = _count(LOCAL / 'images_t1')

if actual < expected:
    print(f'{actual}/{expected} patches in /content/ — copying from Drive...')
    if LOCAL.exists():
        shutil.rmtree(LOCAL)
    LOCAL.mkdir()
    for sub in ['images_t1', 'images_t2', 'masks']:
        subprocess.run(['cp', '-r', str(BASE / 'data/patches' / sub), str(LOCAL)], check=True)
        print(f'  {sub}: {_count(LOCAL / sub)} files')
    subprocess.run(['cp', str(BASE / 'data/patches/split.json'), str(LOCAL)], check=True)
else:
    print(f'Already complete: {actual} patches in /content/')

print(f'Train: {len(SPLIT["train"])}  Val: {len(SPLIT["val"])}')

In [ ]:
# === Cell 4: Dataset ===
# Per-channel normalization stats computed from training set
# T1 and T2 share the same stats (both are Sentinel-2 SR reflectance * 10000)
MEAN = np.array([338, 614, 659, 2600, 2200, 1320], dtype=np.float32)   # rough SR means
STD  = np.array([250, 380, 500, 900,  850,  720],  dtype=np.float32)

aug_train = A.Compose([
    A.RandomCrop(PATCH_SIZE, PATCH_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
])
aug_val = A.Compose([A.CenterCrop(PATCH_SIZE, PATCH_SIZE)])


class ForestDataset(Dataset):
    def __init__(self, names, local_dir, augment):
        self.names   = names
        self.t1_dir  = local_dir / 'images_t1'
        self.t2_dir  = local_dir / 'images_t2'
        self.msk_dir = local_dir / 'masks'
        self.aug     = augment

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        name = self.names[idx]
        t1   = np.load(self.t1_dir  / name).astype(np.float32)   # (6,256,256)
        t2   = np.load(self.t2_dir  / name).astype(np.float32)   # (6,256,256)
        mask = np.load(self.msk_dir / name).astype(np.float32)   # (256,256)

        # Normalize each date
        t1 = (t1 - MEAN[:, None, None]) / (STD[:, None, None] + 1e-6)
        t2 = (t2 - MEAN[:, None, None]) / (STD[:, None, None] + 1e-6)

        # Albumentations expects (H,W,C)
        img  = np.concatenate([t1, t2], axis=0).transpose(1, 2, 0)  # (256,256,12)
        res  = self.aug(image=img, mask=mask)
        img  = res['image'].transpose(2, 0, 1)   # (12,H,W)
        mask = res['mask']

        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)


train_ds = ForestDataset(SPLIT['train'], LOCAL, aug_train)
val_ds   = ForestDataset(SPLIT['val'],   LOCAL, aug_val)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train batches: {len(train_dl)}  Val batches: {len(val_dl)}')

In [ ]:
# === Cell 5: Model — Early Fusion U-Net ===
# 12-channel input (T1+T2 concatenated), ResNet34 encoder, binary output
model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=12,
    classes=1,
    activation=None,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model params: {total_params:.1f}M')

In [ ]:
# === Cell 6: Loss + optimizer ===
class BCEDiceLoss(nn.Module):
    def __init__(self, bce_w=0.5):
        super().__init__()
        self.bce_w  = bce_w
        self.bce    = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0]).to(DEVICE))
        self.dice   = smp.losses.DiceLoss(mode='binary', from_logits=True)

    def forward(self, pred, target):
        return self.bce_w * self.bce(pred, target) + (1 - self.bce_w) * self.dice(pred, target)


criterion = BCEDiceLoss(bce_w=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=1e-6)
print('Loss: 0.5*BCE + 0.5*Dice  |  Adam lr=1e-4  |  CosineAnneal')

In [ ]:
# === Cell 7: Training loop ===
def iou_score(pred_logits, target, threshold=0.5):
    pred  = (pred_logits.sigmoid() > threshold).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum() - inter
    return (inter / (union + 1e-6)).item()


best_iou   = 0.0
ckpt_path  = MODELS_DIR / 'best_forest_disturbance.pth'
history    = []

for epoch in range(1, N_EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss = train_iou = 0.0
    for imgs, masks in tqdm(train_dl, desc=f'Ep {epoch:02d} train', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_iou  += iou_score(preds, masks)

    # --- Validate ---
    model.eval()
    val_loss = val_iou = 0.0
    with torch.no_grad():
        for imgs, masks in val_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds    = model(imgs)
            val_loss += criterion(preds, masks).item()
            val_iou  += iou_score(preds, masks)

    n_tr = len(train_dl); n_va = len(val_dl)
    tl = train_loss/n_tr; ti = train_iou/n_tr
    vl = val_loss/n_va;   vi = val_iou/n_va
    scheduler.step()

    history.append({'epoch': epoch, 'train_loss': tl, 'train_iou': ti,
                    'val_loss': vl, 'val_iou': vi})

    if vi > best_iou:
        best_iou = vi
        torch.save(model.state_dict(), ckpt_path)
        tag = '  ** BEST **'
    else:
        tag = ''

    print(f'Ep {epoch:02d}/{N_EPOCHS}  '
          f'loss={tl:.4f}/{vl:.4f}  IoU={ti:.3f}/{vi:.3f}{tag}')

print(f'\nBest val IoU: {best_iou:.3f}  checkpoint: {ckpt_path}')

In [ ]:
# === Cell 8: Save training history ===
import json, matplotlib.pyplot as plt

with open(MODELS_DIR / 'training_history.json', 'w') as f:
    json.dump(history, f, indent=2)

epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(epochs, [h['train_loss'] for h in history], label='Train')
axes[0].plot(epochs, [h['val_loss']   for h in history], label='Val')
axes[0].set_title('Loss', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, [h['train_iou'] for h in history], label='Train')
axes[1].plot(epochs, [h['val_iou']   for h in history], label='Val')
axes[1].set_title('IoU', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle(f'Training Curves — Best Val IoU = {best_iou:.3f}', fontweight='bold')
plt.tight_layout()
plt.savefig(BASE / 'results/training_curves.png', dpi=100, bbox_inches='tight')
plt.show()
print(f'Checkpoint saved: {ckpt_path}')